# PHOENIX-2014-T CSLR — Pipeline Principale
## DenseNet121 + BiLSTM + CTC con tutte le ottimizzazioni dai paper

Questo notebook orchestra i moduli `.py` del progetto:

| Modulo | Contenuto |
|--------|----------|
| `config.py` | Percorsi e iperparametri |
| `utils.py` | Funzioni di utilità generali |
| `augmentation.py` | Augmentazione TSSI |
| `skeleton.py` | DFS order + generazione TSSI |
| `vocabulary.py` | Vocabolario, merge map, log-prior |
| `dataset.py` | Dataset e collate_fn |
| `model.py` | PoseNetworkCTC (TCN + Attention + BiLSTM) |
| `losses.py` | CTCLossWithEntropy |
| `decoding.py` | Beam search + greedy + bigram |
| `metrics.py` | WER, precision/recall/F1 per gloss |
| `training.py` | run_epoch, two-phase, checkpoint |
| `ensemble.py` | Ensemble decode |
| `optuna_search.py` | Ricerca iperparametri |


## 0 — Setup e Import

In [1]:
import os, gc, sys, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Assicurati che la cartella del progetto sia nel path
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Moduli del progetto
from config import CONFIG, DEVICE, RESULTS_DIR, TSSI_OUTPUT_DIR, ANNOTATIONS_DIR, POSE_DIR
from utils import cleanup_cuda
from skeleton import DFS_ORDER, DFS_UNIQUE, get_dfs_order_phoenix_correct, generate_tssi_optimized
from vocabulary import load_split, build_vocabulary
from dataset import index_pose_files, PHOENIXDatasetContinuos, collate_fn_ctc
from model import PoseNetworkCTC
from losses import CTCLossWithSmoothing
from decoding import greedy_decode_with_prior, beam_search_ctc_optimized
from metrics import (
    compute_wer, summarize_label_distribution,
    compute_word_recognition_metrics, print_word_metrics
)
from training import (
    freeze_backbone, unfreeze_backbone,
    make_optimizer_phase1, make_optimizer_phase2,
    make_scheduler, run_epoch,
    print_epoch_summary, preview_one_batch
)
from ensemble import ensemble_decode

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\u2713 Import completati')

Device: cpu
✓ Import completati


## 1 — Caricamento Dati e Vocabolario

In [ ]:
# Carica CSV
train_df = load_split('train')
dev_df   = load_split('dev')
test_df  = load_split('test')
print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')

# Pipeline vocabolario (merge + OOV + vocab + log-prior)
(
    train_df, dev_df, test_df,
    vocab, g2i, i2g, num_classes, LOG_PRIOR, merge_map
) = build_vocabulary(train_df, dev_df, test_df)

print(f'\nVocabolario: {num_classes} classi')
print(f"  g2i['<blank>'] = {g2i['<blank>']}")
print(f"  g2i['<unk>']   = {g2i['<unk>']}")

## 2 — Indicizzazione Keypoint

In [ ]:
train_kp = index_pose_files('train')
dev_kp   = index_pose_files('dev')
test_kp  = index_pose_files('test')
print(f'Keypoints \u2014 Train: {len(train_kp)} | Dev: {len(dev_kp)} | Test: {len(test_kp)}')

print(f'\nDFS_ORDER: {len(DFS_ORDER)} step, {len(DFS_UNIQUE)} joint unici')

## 3 — Generazione Cache TSSI (skip se già presente)

In [ ]:
for split in ['train', 'dev', 'test']:
    os.makedirs(os.path.join(TSSI_OUTPUT_DIR, split), exist_ok=True)

def cache_tssi(kp_dict, split_name):
    print(f'\nElaborazione {split_name}...')
    for vid_name, kp_path in tqdm(kp_dict.items(), desc=split_name):
        save_path = os.path.join(TSSI_OUTPUT_DIR, split_name, f'{vid_name}.npz')
        if not os.path.exists(save_path):
            kp_array        = np.load(kp_path)
            tssi, seq_len   = generate_tssi_optimized(kp_array)
            np.savez_compressed(save_path, tssi=tssi, seq_len=seq_len)

cache_tssi(train_kp, 'train')
cache_tssi(dev_kp,   'dev')
cache_tssi(test_kp,  'test')
print('\n\u2713 Cache TSSI pronta')

## 4 — Dataset e DataLoader

In [ ]:
train_ds = PHOENIXDatasetContinuos(
    train_df, train_kp, g2i, augment=True,
    tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'train')
)
dev_ds = PHOENIXDatasetContinuos(
    dev_df, dev_kp, g2i, augment=False,
    tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'dev')
)
test_ds = PHOENIXDatasetContinuos(
    test_df, test_kp, g2i, augment=False,
    tssi_dir=os.path.join(TSSI_OUTPUT_DIR, 'test')
)

BS          = CONFIG['batch_size']
NUM_WORKERS = 4

train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc, drop_last=True)
dev_dl   = DataLoader(dev_ds,   batch_size=BS, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc)
test_dl  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn_ctc)

print(f'BS={BS} | Train={len(train_dl)} | Dev={len(dev_dl)} | Test={len(test_dl)}')

## 5 — Modello

In [ ]:
cleanup_cuda()

model = PoseNetworkCTC(
    num_classes=num_classes,
    num_joints=CONFIG['num_joints'],
    hidden_dim=CONFIG['hidden_dim'],
    tcn_blocks=CONFIG['tcn_blocks'],
    lstm_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    drop_path_rate=CONFIG['drop_path_rate'],
    attn_heads=CONFIG['attn_heads'],
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametri totali:    {total_params:,}')
print(f'Parametri trainable: {trainable_params:,}')
print(f'\u2713 PoseNetworkCTC istanziato su {DEVICE}')

## 6 — Debug Pre-Training

In [ ]:
summarize_label_distribution(train_ds, i2g, top_k=20)
print('\nPreview batch train:')
preview_one_batch(model, train_dl, 'train', DEVICE, CONFIG, LOG_PRIOR, i2g)
print('\nPreview batch dev:')
preview_one_batch(model, dev_dl,   'dev',   DEVICE, CONFIG, LOG_PRIOR, i2g)

## 7 — (Opzionale) Ricerca Iperparametri con Optuna

In [ ]:
# Decommentare per eseguire la ricerca
# from optuna_search import run_optuna_search
# best_params = run_optuna_search(
#     train_ds, dev_ds, num_classes, i2g, LOG_PRIOR, n_trials=30
# )
# # Aggiorna CONFIG con i migliori parametri trovati
# CONFIG.update(best_params)
print('Optuna search commentato \u2014 decommenta per eseguire.')

## 8 — Training Two-Phase

In [ ]:
criterion        = CTCLossWithSmoothing(blank=0, smoothing=CONFIG['ctc_smoothing'])
scaler           = GradScaler(enabled=CONFIG['use_amp'] and DEVICE.type == 'cuda')
best_wer         = float('inf')
patience         = 0
history          = {'train_loss': [], 'dev_loss': [], 'dev_wer': [], 'phase': []}
checkpoint_paths = []
global_epoch     = 0

# \u2500\u2500\u2500 FASE 1: backbone congelato \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print('=' * 60)
print(f'FASE 1: {CONFIG["phase1_epochs"]} epoch, BACKBONE CONGELATO')
print('=' * 60)

freeze_backbone(model)
optimizer_p1 = make_optimizer_phase1(model, lr=CONFIG['phase1_lr'],
                                     weight_decay=CONFIG['weight_decay'])
scheduler_p1 = make_scheduler(optimizer_p1, CONFIG['phase1_epochs'], len(train_dl))

for epoch in range(CONFIG['phase1_epochs']):
    global_epoch += 1

    (train_loss, train_wer, train_metrics,
     train_wm, train_pc, train_never, train_low,
     train_wm_unk, train_pc_unk, train_never_unk, train_low_unk) = run_epoch(
        model, train_dl, criterion, optimizer_p1, scaler, scheduler_p1,
        training=True, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    (dev_loss, dev_wer, dev_metrics,
     dev_wm, dev_pc, dev_never, dev_low,
     dev_wm_unk, dev_pc_unk, dev_never_unk, dev_low_unk) = run_epoch(
        model, dev_dl, criterion, optimizer_p1, scaler, scheduler_p1,
        training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    print_epoch_summary(global_epoch, train_metrics, train_loss, train_wer, training=True)
    print_epoch_summary(global_epoch, dev_metrics,   dev_loss,   dev_wer,   training=False)
    print_word_metrics(dev_wm, dev_pc, dev_never, dev_low,
                       summary_unk=dev_wm_unk, per_class_unk=dev_pc_unk,
                       never_predicted_unk=dev_never_unk, low_recall_unk=dev_low_unk,
                       epoch=global_epoch, split='VAL')

    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_wer'].append(dev_wer)
    history['phase'].append(1)

    print(f'[FASE 1] Ep {epoch+1:2d}/{CONFIG["phase1_epochs"]} '
          f'(Global {global_epoch}) | '
          f'Loss {train_loss:.4f}/{dev_loss:.4f} | '
          f'WER {train_wer*100:.1f}%/{dev_wer*100:.1f}%')

    if dev_wer < best_wer:
        best_wer = dev_wer
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pth'))
        print(f'  \u2192 Nuovo best WER: {best_wer*100:.2f}%')

print(f'\n\u2713 Fase 1 completata. Best WER: {best_wer*100:.2f}%')

In [ ]:
# \u2500\u2500\u2500 FASE 2: tutto sbloccato, LR differenziale \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
phase2_epochs = CONFIG['num_epochs'] - CONFIG['phase1_epochs']
print('\n' + '=' * 60)
print(f'FASE 2: {phase2_epochs} epoch, TUTTO SBLOCCATO (LR differenziale)')
print('=' * 60)

unfreeze_backbone(model)
optimizer_p2 = make_optimizer_phase2(
    model,
    lr_backbone=CONFIG['phase2_lr_backbone'],
    lr_head=CONFIG['phase2_lr_head'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler_p2 = make_scheduler(optimizer_p2, phase2_epochs, len(train_dl))

for epoch in range(phase2_epochs):
    global_epoch += 1

    (train_loss, train_wer, train_metrics,
     train_wm, train_pc, train_never, train_low,
     train_wm_unk, train_pc_unk, train_never_unk, train_low_unk) = run_epoch(
        model, train_dl, criterion, optimizer_p2, scaler, scheduler_p2,
        training=True, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    (dev_loss, dev_wer, dev_metrics,
     dev_wm, dev_pc, dev_never, dev_low,
     dev_wm_unk, dev_pc_unk, dev_never_unk, dev_low_unk) = run_epoch(
        model, dev_dl, criterion, optimizer_p2, scaler, scheduler_p2,
        training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

    print_epoch_summary(global_epoch, train_metrics, train_loss, train_wer, training=True)
    print_epoch_summary(global_epoch, dev_metrics,   dev_loss,   dev_wer,   training=False)
    print_word_metrics(dev_wm, dev_pc, dev_never, dev_low,
                       summary_unk=dev_wm_unk, per_class_unk=dev_pc_unk,
                       never_predicted_unk=dev_never_unk, low_recall_unk=dev_low_unk,
                       epoch=global_epoch, split='VAL')

    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_wer'].append(dev_wer)
    history['phase'].append(2)

    print(f'[FASE 2] Ep {epoch+1:2d}/{phase2_epochs} '
          f'(Global {global_epoch}) | '
          f'Loss {train_loss:.4f}/{dev_loss:.4f} | '
          f'WER {train_wer*100:.1f}%/{dev_wer*100:.1f}%')

    # Salva checkpoint per ensemble
    ckpt_path = os.path.join(RESULTS_DIR, f'checkpoint_ep{global_epoch:03d}.pth')
    torch.save(model.state_dict(), ckpt_path)
    checkpoint_paths.append((ckpt_path, dev_wer))
    if len(checkpoint_paths) > CONFIG['keep_last_n_checkpoints']:
        old_path, _ = checkpoint_paths.pop(0)
        if os.path.exists(old_path):
            os.remove(old_path)

    if dev_wer < best_wer:
        best_wer = dev_wer
        patience  = 0
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, 'best_model.pth'))
        print(f'  \u2192 Nuovo best WER: {best_wer*100:.2f}%')
    else:
        patience += 1
        if patience >= CONFIG['early_stopping_patience']:
            print(f'Early stopping a epoch {global_epoch}')
            break

    cleanup_cuda()

print(f'\n\u2713 Training completato. Best WER: {best_wer*100:.2f}%')

## 9 — Model Ensemble (Deep Sign §6.5)

In [ ]:
N_ensemble = CONFIG['ensemble_n']
best_ckpts = sorted(checkpoint_paths, key=lambda x: x[1])[:N_ensemble]
ensemble_paths = [p for p, _ in best_ckpts]

print(f'Checkpoint selezionati per ensemble (migliori {N_ensemble} per WER dev):')
for p, w in best_ckpts:
    print(f'  {os.path.basename(p)}  WER dev={w*100:.2f}%')

ensemble_dev_wer = None
if len(ensemble_paths) >= 2:
    ensemble_dev_wer = ensemble_decode(
        ensemble_paths, dev_dl, DEVICE, CONFIG, LOG_PRIOR, num_classes
    )
    print(f'\n\U0001f4ca Ensemble WER (dev):  {ensemble_dev_wer*100:.2f}%')
    print(f'\U0001f4ca Best single WER (dev): {best_wer*100:.2f}%')
    print(f'\U0001f4ca Guadagno ensemble:   {(best_wer - ensemble_dev_wer)*100:.2f}% abs')
else:
    print('\u26a0 Non abbastanza checkpoint per ensemble (servono \u2265 2)')

## 10 — Valutazione sul Test Set

In [ ]:
print('=' * 60 + '\nTEST SET EVALUATION\n' + '=' * 60)

best_path = os.path.join(RESULTS_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

(test_loss, test_wer_single, test_metrics,
 test_wm, test_pc, test_never, test_low,
 test_wm_unk, test_pc_unk, test_never_unk, test_low_unk) = run_epoch(
    model, test_dl, criterion, None, scaler, None,
    training=False, device=DEVICE, config=CONFIG, log_prior=LOG_PRIOR, i2g=i2g)

print_word_metrics(test_wm, test_pc, test_never, test_low,
                   summary_unk=test_wm_unk, per_class_unk=test_pc_unk,
                   never_predicted_unk=test_never_unk, low_recall_unk=test_low_unk,
                   split='TEST')

print(f'\U0001f4ca Test WER (singolo): {test_wer_single*100:.2f}%')

test_wer_ensemble = None
if len(ensemble_paths) >= 2:
    test_wer_ensemble = ensemble_decode(
        ensemble_paths, test_dl, DEVICE, CONFIG, LOG_PRIOR, num_classes
    )
    print(f'\U0001f4ca Test WER (ensemble {len(ensemble_paths)} modelli): {test_wer_ensemble*100:.2f}%')

# Salva risultati
results = {
    'config':            CONFIG,
    'best_dev_wer':      float(best_wer),
    'test_wer_single':   float(test_wer_single),
    'test_wer_ensemble': float(test_wer_ensemble) if test_wer_ensemble is not None else None,
    'vocab_size':        int(num_classes),
    'history':           history,
}
with open(os.path.join(RESULTS_DIR, 'training_results.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('\n\u2705 Risultati salvati in training_results.json')

## 11 — (Opzionale) Bigram LM Rescoring

In [ ]:
from collections import defaultdict
from decoding import greedy_decode_with_bigram

# Costruisci bigram dal train set
bigram_counts = defaultdict(lambda: defaultdict(int))
for seq in train_df['orth'].fillna(''):
    ids = [g2i.get(t) for t in str(seq).strip().upper().split() if g2i.get(t) is not None]
    for a, b in zip(ids, ids[1:]):
        bigram_counts[a][b] += 1

log_bigram = {
    prev: {nxt: np.log(cnt / total)
           for nxt, cnt in nexts.items()}
    for prev, nexts in bigram_counts.items()
    for total in [sum(nexts.values())]
}

# Ricerca alpha ottimale sul dev set
print('Ricerca alpha bigram su dev set...')
best_alpha, best_alpha_wer = 0.0, float('inf')
model.eval()
for alpha in [0.0, 0.1, 0.2, 0.3, 0.5]:
    all_refs, all_hyps = [], []
    with torch.no_grad():
        for tssies, targets, input_lengths, target_lengths in dev_dl:
            tssies = tssies.to(DEVICE)
            lp     = model(tssies).permute(1, 0, 2).float().cpu()
            decoded = greedy_decode_with_bigram(lp, beta=CONFIG['prior_beta'],
                                                log_prior=LOG_PRIOR.cpu(),
                                                log_bigram=log_bigram, alpha=alpha)
            all_hyps.extend(decoded)
            refs, offset = [], 0
            for tlen in target_lengths.tolist():
                all_refs.append(targets[offset:offset+tlen].tolist())
                offset += tlen
    wer, _ = compute_wer(all_refs, all_hyps)
    print(f'  alpha={alpha:.1f} \u2192 Dev WER {wer*100:.2f}%')
    if wer < best_alpha_wer:
        best_alpha_wer, best_alpha = wer, alpha

print(f'\u2713 Best alpha: {best_alpha} (WER {best_alpha_wer*100:.2f}%)')
CONFIG['bigram_alpha'] = best_alpha

## 12 — Visualizzazione Training History

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_all  = range(1, len(history['train_loss']) + 1)
phase1_end  = CONFIG['phase1_epochs']

# Loss
axes[0].axvspan(1, phase1_end, alpha=0.08, color='blue',  label='Fase 1 (frozen)')
axes[0].axvspan(phase1_end, len(history['train_loss']), alpha=0.08, color='green', label='Fase 2 (full)')
axes[0].plot(epochs_all, history['train_loss'], label='Train Loss', alpha=0.8)
axes[0].plot(epochs_all, history['dev_loss'],   label='Dev Loss',   alpha=0.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CTC Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# WER
axes[1].axvspan(1, phase1_end, alpha=0.08, color='blue')
axes[1].axvspan(phase1_end, len(history['dev_wer']), alpha=0.08, color='green')
axes[1].plot(epochs_all, [w * 100 for w in history['dev_wer']],
             label='Dev WER (%)', color='red', alpha=0.8)
axes[1].axhline(best_wer * 100, color='green', linestyle='--',
                label=f'Best single: {best_wer*100:.1f}%')
if test_wer_ensemble is not None:
    axes[1].axhline(test_wer_ensemble * 100, color='purple', linestyle=':',
                    label=f'Ensemble test: {test_wer_ensemble*100:.1f}%')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('WER (%)')
axes[1].set_title('Word Error Rate (Validation)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 60)
print('RIEPILOGO FINALE')
print('=' * 60)
print(f'Vocabolario: {num_classes} classi')
print(f'Best dev WER:       {best_wer*100:.2f}%')
print(f'Test WER (singolo): {test_wer_single*100:.2f}%')
if test_wer_ensemble is not None:
    print(f'Test WER (ensemble): {test_wer_ensemble*100:.2f}%')
print('=' * 60)